In [1]:
from pathlib import Path
import sys
import time
import heapq
from collections import deque

In [2]:
AIPY_DIR = Path("/home/mikolaj/work/inteligencja_obliczeniowa/lab02/aipython/aipython")

if str(AIPY_DIR) not in sys.path:
    sys.path.insert(0, str(AIPY_DIR))

from stripsProblem import Strips, STRIPS_domain, Planning_problem
from stripsForwardPlanner import Forward_STRIPS
from searchMPP import SearcherMPP

In [3]:
BLOCKS = ["a", "b", "c", "d"]
TABLES = ["t1", "t2", "t3"]
SUPPORTS = BLOCKS + TABLES

def on_feat(x):
    return f"on_{x}"

def clear_feat(x):
    return f"clear_{x}"

def empty_feat(t):
    return f"empty_{t}"

In [4]:
def complete_state(assignments, blocks=BLOCKS, tables=TABLES):
    s = {}
    for b in blocks:
        s[on_feat(b)] = None
        s[clear_feat(b)] = False
    for t in tables:
        s[empty_feat(t)] = True
    s.update(assignments)
    return s

In [5]:
def make_move_action(x, y, z):
    name = f"move({x},{y},{z})"

    pre = {
        on_feat(x): y,
        clear_feat(x): True,
    }

    if z in BLOCKS:
        pre[clear_feat(z)] = True
    else:
        pre[empty_feat(z)] = True

    eff = {
        on_feat(x): z,
    }

    if y in BLOCKS:
        eff[clear_feat(y)] = True
    else:
        eff[empty_feat(y)] = True

    if z in BLOCKS:
        eff[clear_feat(z)] = False
    else:
        eff[empty_feat(z)] = False

    return Strips(name, pre, eff, cost=1)

In [6]:
def make_feature_domains(blocks=BLOCKS, tables=TABLES):
    feats_vals = {}

    for b in blocks:
        feats_vals[on_feat(b)] = set(blocks + tables) - {b}
        feats_vals[clear_feat(b)] = {True, False}

    for t in tables:
        feats_vals[empty_feat(t)] = {True, False}

    return feats_vals

In [7]:
def make_blocks_domain(blocks=BLOCKS, tables=TABLES):
    actions = []
    supports = blocks + tables

    for x in blocks:
        for y in supports:
            if y == x:
                continue
            for z in supports:
                if z == x or z == y:
                    continue
                action = make_move_action(x, y, z)
                actions.append(action)

    feats_vals = make_feature_domains(blocks, tables)
    return STRIPS_domain(feats_vals, actions)

domain = make_blocks_domain()
len(domain.actions)

120

In [8]:
def state_to_frozenset(state_dict):
    return frozenset(state_dict.items())

def frozenset_to_state(fs):
    return dict(fs)

In [9]:
def valid_state(state, blocks=BLOCKS, tables=TABLES):
    # 1) każdy klocek ma dokładnie jedno podparcie
    supports_count = {b: 0 for b in blocks}
    for b in blocks:
        if state[on_feat(b)] in SUPPORTS:
            supports_count[b] += 1
    if any(v != 1 for v in supports_count.values()):
        return False

    # 2) brak cykli w relacji on
    for b in blocks:
        seen = set()
        cur = b
        while True:
            sup = state[on_feat(cur)]
            if sup in TABLES:
                break
            if sup in seen:
                return False
            seen.add(sup)
            cur = sup

    # 3) clear(x) zgodne z tym, czy coś leży na x
    for x in blocks:
        someone_on_x = any(state[on_feat(b)] == x for b in blocks if b != x)
        if state[clear_feat(x)] == someone_on_x:
            return False

    # 4) empty(t) zgodne z tym, czy coś stoi bezpośrednio na stole t
    for t in tables:
        someone_on_t = any(state[on_feat(b)] == t for b in blocks)
        if state[empty_feat(t)] == someone_on_t:
            return False

    return True

In [10]:
def apply_action_to_state(state, action):
    new_state = dict(state)
    for f, v in action.effects.items():
        new_state[f] = v
    return new_state

In [11]:
# Problem 1: przeniesienie wieży a-b-c z t1 na t2, d stoi osobno
problem1_init = complete_state({
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "t1",
    on_feat("d"): "t3",
    clear_feat("a"): True,
    clear_feat("b"): False,
    clear_feat("c"): False,
    clear_feat("d"): True,
    empty_feat("t1"): False,
    empty_feat("t2"): True,
    empty_feat("t3"): False,
})

problem1_goal = {
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "t2",
}

problem1 = Planning_problem(domain, problem1_init, problem1_goal)

In [12]:
# Problem 2: odwrócenie wieży a-b-c na c-b-a i przeniesienie na t2
problem2_init = complete_state({
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "t1",
    on_feat("d"): "t3",
    clear_feat("a"): True,
    clear_feat("b"): False,
    clear_feat("c"): False,
    clear_feat("d"): True,
    empty_feat("t1"): False,
    empty_feat("t2"): True,
    empty_feat("t3"): False,
})

problem2_goal = {
    on_feat("a"): "d",
    on_feat("d"): "c",
    on_feat("c"): "b",
    on_feat("b"): "t2",
}

problem2 = Planning_problem(domain, problem2_init, problem2_goal)

In [13]:
# Problem 3: wariant Sussman-like
problem3_init = complete_state({
    on_feat("a"): "t1",
    on_feat("b"): "t2",
    on_feat("c"): "a",
    on_feat("d"): "t3",
    clear_feat("a"): False,
    clear_feat("b"): True,
    clear_feat("c"): True,
    clear_feat("d"): True,
    empty_feat("t1"): False,
    empty_feat("t2"): False,
    empty_feat("t3"): False,
})

problem3_goal = {
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "t2",
}

problem3 = Planning_problem(domain, problem3_init, problem3_goal)

In [14]:
problems = {
    "problem1": problem1,
    "problem2": problem2,
    "problem3": problem3,
}

for name, p in problems.items():
    print(name, valid_state(p.initial_state))

problem1 True
problem2 True
problem3 True


In [15]:
def get_problem_domain(problem):
    if hasattr(problem, "prob_domain"):
        return problem.prob_domain
    if hasattr(problem, "domain"):
        return problem.domain
    raise AttributeError(f"Nie znalazłem domeny w Planning_problem. Atrybuty: {dir(problem)}")

In [16]:
def count_reachable_states(problem, timeout_seconds=5):
    start = time.perf_counter()

    domain = get_problem_domain(problem)
    init_fs = state_to_frozenset(problem.initial_state)

    visited = {init_fs}
    q = deque([problem.initial_state])

    while q and time.perf_counter() - start < timeout_seconds:
        state = q.popleft()

        for action in domain.actions:
            ok = True
            for f, v in action.preconds.items():
                if state[f] != v:
                    ok = False
                    break
            if not ok:
                continue

            ns = apply_action_to_state(state, action)
            if not valid_state(ns):
                continue

            fs = state_to_frozenset(ns)
            if fs not in visited:
                visited.add(fs)
                q.append(ns)

    return len(visited)

for name, p in problems.items():
    print(name, count_reachable_states(p, timeout_seconds=5))

problem1 360


problem2 360
problem3 360


In [17]:
# Forward planning w AIPython
def solve_with_forward_planner(problem):
    planner = Forward_STRIPS(problem)
    searcher = SearcherMPP(planner)
    path = searcher.search()
    return planner, searcher, path

In [18]:
def extract_actions_from_path(path):
    if path is None:
        return None

    actions = []
    cur = path
    while hasattr(cur, "arc") and cur.arc is not None:
        actions.append(cur.arc.action)
        cur = cur.initial
    actions.reverse()
    return actions

In [19]:
forward_results = {}

for name, p in problems.items():
    start = time.perf_counter()
    planner, searcher, path = solve_with_forward_planner(p)
    elapsed = time.perf_counter() - start
    plan = extract_actions_from_path(path)

    forward_results[name] = {
        "plan": plan,
        "time_s": elapsed,
        "expanded": getattr(searcher, "num_expanded", None),
    }

    print("=" * 80)
    print(name)
    print("time_s:", round(elapsed, 4))
    print("expanded:", getattr(searcher, "num_expanded", None))
    print("plan_len:", None if plan is None else len(plan))
    print("plan:", plan)

Solution: {'on_a': 'b', 'clear_a': True, 'on_b': 'c', 'clear_b': False, 'on_c': 't1', 'clear_c': False, 'on_d': 't3', 'clear_d': True, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(a,b,d)--> {'on_a': 'd', 'clear_a': True, 'on_b': 'c', 'clear_b': True, 'on_c': 't1', 'clear_c': False, 'on_d': 't3', 'clear_d': False, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(b,c,a)--> {'on_a': 'd', 'clear_a': False, 'on_b': 'a', 'clear_b': True, 'on_c': 't1', 'clear_c': True, 'on_d': 't3', 'clear_d': False, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(c,t1,t2)--> {'on_a': 'd', 'clear_a': False, 'on_b': 'a', 'clear_b': True, 'on_c': 't2', 'clear_c': True, 'on_d': 't3', 'clear_d': False, 'empty_t1': True, 'empty_t2': False, 'empty_t3': False}
   --move(b,a,c)--> {'on_a': 'd', 'clear_a': True, 'on_b': 'c', 'clear_b': True, 'on_c': 't2', 'clear_c': False, 'on_d': 't3', 'clear_d': False, 'empty_t1': True, 'empty_t2': False, 'empty_t3': False}
   

 144 paths have been expanded and 290 paths remain in the frontier
problem1
time_s: 0.0836
expanded: 144
plan_len: 5
plan: [move(a,b,d), move(b,c,a), move(c,t1,t2), move(b,a,c), move(a,d,b)]
Solution: {'on_a': 'b', 'clear_a': True, 'on_b': 'c', 'clear_b': False, 'on_c': 't1', 'clear_c': False, 'on_d': 't3', 'clear_d': True, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(a,b,d)--> {'on_a': 'd', 'clear_a': True, 'on_b': 'c', 'clear_b': True, 'on_c': 't1', 'clear_c': False, 'on_d': 't3', 'clear_d': False, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(b,c,t2)--> {'on_a': 'd', 'clear_a': True, 'on_b': 't2', 'clear_b': True, 'on_c': 't1', 'clear_c': True, 'on_d': 't3', 'clear_d': False, 'empty_t1': False, 'empty_t2': False, 'empty_t3': False}
   --move(c,t1,b)--> {'on_a': 'd', 'clear_a': True, 'on_b': 't2', 'clear_b': False, 'on_c': 'b', 'clear_c': True, 'on_d': 't3', 'clear_d': False, 'empty_t1': True, 'empty_t2': False, 'empty_t3': False}
   --move(a,

In [20]:
# Heurystyka:
# liczba niespełnionych relacji celu + kara za "zły support" dla klocków
def blocks_heuristic(state, goal):
    h = 0
    for f, v in goal.items():
        if state[f] != v:
            h += 1

    # dodatkowa lekka kara za klocki, które stoją na złym supportcie
    for b in BLOCKS:
        feat = on_feat(b)
        if feat in goal and state[feat] != goal[feat]:
            h += 1

    return h

In [21]:
def astar_problem(problem, heuristic_fn, timeout_seconds=300):
    start_time = time.perf_counter()

    domain = get_problem_domain(problem)
    start_state = problem.initial_state
    start_fs = state_to_frozenset(start_state)

    open_heap = []
    counter = 0

    g_score = {start_fs: 0}
    start_h = heuristic_fn(start_state, problem.goal)
    heapq.heappush(open_heap, (start_h, counter, start_state, []))

    closed = set()
    expanded = 0

    while open_heap:
        if time.perf_counter() - start_time > timeout_seconds:
            return None, time.perf_counter() - start_time, expanded

        f, _, state, plan = heapq.heappop(open_heap)
        state_fs = state_to_frozenset(state)

        if state_fs in closed:
            continue
        closed.add(state_fs)
        expanded += 1

        goal_ok = True
        for feat, val in problem.goal.items():
            if state[feat] != val:
                goal_ok = False
                break
        if goal_ok:
            return plan, time.perf_counter() - start_time, expanded

        for action in domain.actions:
            ok = True
            for feat, val in action.preconds.items():
                if state[feat] != val:
                    ok = False
                    break
            if not ok:
                continue

            ns = apply_action_to_state(state, action)
            if not valid_state(ns):
                continue

            ns_fs = state_to_frozenset(ns)
            tentative_g = g_score[state_fs] + 1

            if ns_fs not in g_score or tentative_g < g_score[ns_fs]:
                g_score[ns_fs] = tentative_g
                counter += 1
                h = heuristic_fn(ns, problem.goal)
                heapq.heappush(open_heap, (tentative_g + h, counter, ns, plan + [action.name]))

    return None, time.perf_counter() - start_time, expanded

In [22]:
heur_results = {}

for name, p in problems.items():
    plan, elapsed, expanded = astar_problem(p, blocks_heuristic, timeout_seconds=300)

    heur_results[name] = {
        "plan": plan,
        "time_s": elapsed,
        "expanded": expanded,
    }

    print("=" * 80)
    print(name)
    print("time_s:", round(elapsed, 4))
    print("expanded:", expanded)
    print("plan_len:", None if plan is None else len(plan))
    print("plan:", plan)

problem1
time_s: 0.0142
expanded: 16
plan_len: 5
plan: ['move(a,b,d)', 'move(b,c,a)', 'move(c,t1,t2)', 'move(b,a,c)', 'move(a,d,b)']
problem2
time_s: 0.0071
expanded: 11
plan_len: 6
plan: ['move(a,b,d)', 'move(b,c,t2)', 'move(c,t1,b)', 'move(a,d,t1)', 'move(d,t3,c)', 'move(a,t1,d)']
problem3
time_s: 0.0028
expanded: 9
plan_len: 4
plan: ['move(b,t2,d)', 'move(c,a,t2)', 'move(b,d,c)', 'move(a,t1,b)']


In [23]:
heur_results = {}

for name, p in problems.items():
    plan, elapsed, expanded = astar_problem(p, blocks_heuristic, timeout_seconds=300)

    heur_results[name] = {
        "plan": plan,
        "time_s": elapsed,
        "expanded": expanded,
    }

    print("=" * 80)
    print(name)
    print("time_s:", round(elapsed, 4))
    print("expanded:", expanded)
    print("plan_len:", None if plan is None else len(plan))
    print("plan:", plan)

problem1
time_s: 0.0035
expanded: 16
plan_len: 5
plan: ['move(a,b,d)', 'move(b,c,a)', 'move(c,t1,t2)', 'move(b,a,c)', 'move(a,d,b)']
problem2
time_s: 0.0028
expanded: 11
plan_len: 6
plan: ['move(a,b,d)', 'move(b,c,t2)', 'move(c,t1,b)', 'move(a,d,t1)', 'move(d,t3,c)', 'move(a,t1,d)']
problem3
time_s: 0.0018
expanded: 9
plan_len: 4
plan: ['move(b,t2,d)', 'move(c,a,t2)', 'move(b,d,c)', 'move(a,t1,b)']


In [24]:
import pandas as pd

rows = []
for name in problems:
    rows.append({
        "problem": name,
        "reachable_states_5s": count_reachable_states(problems[name], 5),
        "forward_plan_len": None if forward_results[name]["plan"] is None else len(forward_results[name]["plan"]),
        "forward_time_s": round(forward_results[name]["time_s"], 4),
        "forward_expanded": forward_results[name]["expanded"],
        "heur_plan_len": None if heur_results[name]["plan"] is None else len(heur_results[name]["plan"]),
        "heur_time_s": round(heur_results[name]["time_s"], 4),
        "heur_expanded": heur_results[name]["expanded"],
        "heur_plan": heur_results[name]["plan"],
    })

df = pd.DataFrame(rows)
df

,problem,reachable_states_5s,forward_plan_len,forward_time_s,forward_expanded,heur_plan_len,heur_time_s,heur_expanded,heur_plan
0,problem1,360,5,0.0836,144,5,0.0035,16,"[move(a,b,d), move(b,c,a), move(c,t1,t2), move..."
1,problem2,360,6,0.1431,177,6,0.0028,11,"[move(a,b,d), move(b,c,t2), move(c,t1,b), move..."
2,problem3,360,4,0.0270,62,4,0.0018,9,"[move(b,t2,d), move(c,a,t2), move(b,d,c), move..."


In [25]:
# Ładny wydruk planów do sprawozdania
for name in problems:
    print("=" * 100)
    print(name)
    print("Forward planning:")
    if forward_results[name]["plan"] is not None:
        for i, a in enumerate(forward_results[name]["plan"], 1):
            print(f"{i}. {a}")
    else:
        print("brak rozwiązania")

    print("\nZ heurystyką:")
    if heur_results[name]["plan"] is not None:
        for i, a in enumerate(heur_results[name]["plan"], 1):
            print(f"{i}. {a}")
    else:
        print("brak rozwiązania")

    print("\nCzas z heurystyką:", round(heur_results[name]["time_s"], 4), "s")

problem1
Forward planning:
1. move(a,b,d)
2. move(b,c,a)
3. move(c,t1,t2)
4. move(b,a,c)
5. move(a,d,b)

Z heurystyką:
1. move(a,b,d)
2. move(b,c,a)
3. move(c,t1,t2)
4. move(b,a,c)
5. move(a,d,b)

Czas z heurystyką: 0.0035 s
problem2
Forward planning:
1. move(a,b,d)
2. move(b,c,t2)
3. move(c,t1,b)
4. move(a,d,t1)
5. move(d,t3,c)
6. move(a,t1,d)

Z heurystyką:
1. move(a,b,d)
2. move(b,c,t2)
3. move(c,t1,b)
4. move(a,d,t1)
5. move(d,t3,c)
6. move(a,t1,d)

Czas z heurystyką: 0.0028 s
problem3
Forward planning:
1. move(b,t2,d)
2. move(c,a,t2)
3. move(b,d,c)
4. move(a,t1,b)

Z heurystyką:
1. move(b,t2,d)
2. move(c,a,t2)
3. move(b,d,c)
4. move(a,t1,b)

Czas z heurystyką: 0.0018 s


In [26]:
# Krótka kontrola warunku z treści zadania:
for name in problems:
    rs = count_reachable_states(problems[name], 5)
    fp = forward_results[name]["plan"]
    hp = heur_results[name]["plan"]
    print(name, {
        "reachable_states_5s": rs,
        "forward_len": None if fp is None else len(fp),
        "heur_len": None if hp is None else len(hp),
    })

problem1 {'reachable_states_5s': 360, 'forward_len': 5, 'heur_len': 5}
problem2 {'reachable_states_5s': 360, 'forward_len': 6, 'heur_len': 6}
problem3 {'reachable_states_5s': 360, 'forward_len': 4, 'heur_len': 4}
